# Hugging Face as a Data and ML Model Hub


**Learning goal**
By the end, students understand:

We can select a dataset/task, choose a pretrained **Hugging Face** model, and reuse it without training.


In this notebook, we will build a tiny AI project without training a model from scratch.


## Tiny Project: Student Feedback Text Classifier

Imagine we collect short feedback messages from students after a lecture.

We want to automatically classify whether the feedback is:

- Positive
- Negative
- Nutrual

Instead of training a sentiment model ourselves, we will reuse a pretrained model from Hugging Face.

We will use this model for the text classification task from Hugging Face: "distilbert-base-uncased-finetuned-sst-2-english"

Find the model in the Hugging Face website, and click on `use this model`. Copy the code and paste it in your project to use this model.

In [1]:
from transformers import pipeline
import pandas as pd

## Create a tiny feedback dataset

In [2]:
feedback = [
    "The lecture was clear and the demo was useful.",
    "I did not understand the explanation about APIs.",
    "The examples helped me understand the topic.",
    "The pace was too fast and confusing.",
    "I liked the practical part of the class.",
    "I need more explanation and more practice.",
]

df = pd.DataFrame({"feedback": feedback})
df

,feedback
0,The lecture was clear and the demo was useful.
1,I did not understand the explanation about APIs.
2,The examples helped me understand the topic.
3,The pace was too fast and confusing.
4,I liked the practical part of the class.
5,I need more explanation and more practice.


## Load pretrained model from HF

In [3]:
model_name = "distilbert-base-uncased-finetuned-sst-2-english"

classifier = pipeline(
    task="text-classification",
    model=model_name
)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

## Try one prediction

In [4]:
text = "The demo was clear and useful."

prediction = classifier(text)
prediction

[{'label': 'POSITIVE', 'score': 0.9995637536048889}]

## Predict all feedback messages

In [5]:
predictions = classifier(df["feedback"].tolist())

predictions

[{'label': 'POSITIVE', 'score': 0.9932767748832703},
 {'label': 'NEGATIVE', 'score': 0.9997149109840393},
 {'label': 'POSITIVE', 'score': 0.9993225336074829},
 {'label': 'NEGATIVE', 'score': 0.9996639490127563},
 {'label': 'POSITIVE', 'score': 0.9997602105140686},
 {'label': 'NEGATIVE', 'score': 0.9995751976966858}]

## Add the result to a dataframe

In [6]:
df["label"] = [p["label"] for p in predictions]
df["confidence"] = [round(p["score"], 3) for p in predictions]

df

,feedback,label,confidence
0,The lecture was clear and the demo was useful.,POSITIVE,0.993
1,I did not understand the explanation about APIs.,NEGATIVE,1.000
2,The examples helped me understand the topic.,POSITIVE,0.999
3,The pace was too fast and confusing.,NEGATIVE,1.000
4,I liked the practical part of the class.,POSITIVE,1.000
5,I need more explanation and more practice.,NEGATIVE,1.000


## Make it reusable

In [7]:
def classify_feedback(text):
    result = classifier(text)[0]
    return {
        "text": text,
        "label": result["label"],
        "confidence": round(result["score"], 3)
    }

In [8]:
classify_feedback("This class was interesting, but I need more examples.")

{'text': 'This class was interesting, but I need more examples.',
 'label': 'NEGATIVE',
 'confidence': 0.937}

## We can try another model

In [9]:
from transformers import pipeline
import pandas as pd

model_name = "cardiffnlp/twitter-roberta-base-sentiment-latest"

classifier = pipeline(
    task="sentiment-analysis",
    model=model_name
)


texts = [
    "The lecture was clear and useful.",
    "I did not understand the explanation.",
    "The lecture was about Hugging Face models.",
    "The class started at 10 AM.",
    "The topic was interesting, but I need more practice."
]

classifier(texts)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[{'label': 'positive', 'score': 0.9573295712471008},
 {'label': 'negative', 'score': 0.6574666500091553},
 {'label': 'neutral', 'score': 0.8709646463394165},
 {'label': 'neutral', 'score': 0.9334155321121216},
 {'label': 'positive', 'score': 0.9069417715072632}]

In [10]:
predictions = classifier(df["feedback"].tolist())

df["label"] = [p["label"] for p in predictions]
df["confidence"] = [round(p["score"], 3) for p in predictions]

df

,feedback,label,confidence
0,The lecture was clear and the demo was useful.,positive,0.948
1,I did not understand the explanation about APIs.,negative,0.561
2,The examples helped me understand the topic.,neutral,0.558
3,The pace was too fast and confusing.,negative,0.715
4,I liked the practical part of the class.,positive,0.952
5,I need more explanation and more practice.,neutral,0.754


> This is why choosing the right model matters. The first model only knows positive and negative. If our project needs neutral feedback, we must select a model trained with a neutral class.

## Takeaways:

We built a text classifier by reusing a pretrained model.

We did not:

- collect a large dataset
- train a neural network
- use GPUs
- tune model parameters

We only selected a task, selected a model, and ran inference.